
# **Cell 1: Environment Setup & Kaggle Secrets Authentication**
*Installs optimized dependencies and authenticates Hugging Face via Kaggle Secrets.*

In [6]:
import os
import glob

# Search for the processed tar file anywhere in the Kaggle input directory
found_files = glob.glob("/kaggle/input/**/brats2021_processed.tar", recursive=True)

if found_files:
    dataset_path = found_files[0]
    size_gb = os.path.getsize(dataset_path) / (1024**3)

    print(f"✅ Success! Dataset successfully mounted.")
    print(f"📁 Location: {dataset_path}")
    print(f"📦 File size: {size_gb:.2f} GB")
else:
    print("❌ Dataset not found. Make sure you clicked '+ Add Input' and selected your preprocessing notebook!")

✅ Success! Dataset successfully mounted.
📁 Location: /kaggle/input/notebooks/aliqaiser1123/brats2021-task1-preprocessing/brats2021_processed.tar
📦 File size: 14.72 GB


In [7]:
# We added torchvision and torchaudio to the upgrade list to keep them synced with torch
!pip install -q -U torch torchvision torchaudio transformers bitsandbytes peft accelerate nibabel scipy tqdm huggingface_hub monai safetensors

import gc
import os
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    user_secrets = UserSecretsClient()
    try:
        hf_token = user_secrets.get_secret("HF-TOKEN")
    except Exception:
        hf_token = user_secrets.get_secret("huggingface")
    login(token=hf_token)
    print("✅ Authenticated with Hugging Face!")
except Exception as e:
    print(f"⚠️ Auth Warning: {e}. Ensure HF-TOKEN is configured in Kaggle Secrets.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device} | CUDA Available: {torch.cuda.is_available()}")

✅ Authenticated with Hugging Face!
Compute device: cuda | CUDA Available: True


# **Cell 2: Zero-Disk Virtual Dataset & Safe Collator**
*Streams patient volumes into temporary RAM (`/tmp`), normalizes intensities, generates clinical QA pairs, and purges RAM immediately after loading.*

In [8]:
import glob
import tarfile
import shutil
import numpy as np
import nibabel as nib
import scipy.ndimage as ndimage
from torch.utils.data import Dataset, DataLoader

class BraTS_VLM_VirtualDataset(Dataset):
    def __init__(self, tar_file_path, target_shape=(96, 96, 96)):
        self.tar_file_path = tar_file_path
        self.target_shape = target_shape
        self.patient_catalog = {}

        with tarfile.open(self.tar_file_path, "r") as tf:
            for member in tf.getmembers():
                parts = member.name.split('/')
                if len(parts) > 1 and parts[1].startswith("BraTS2021_"):
                    pid = parts[1]
                    if pid not in self.patient_catalog:
                        self.patient_catalog[pid] = []
                    self.patient_catalog[pid].append(member)

        self.patient_ids = sorted(list(self.patient_catalog.keys()))

    def __len__(self):
        return len(self.patient_ids)

    def _resize_and_normalize(self, volume):
        factors = [t / s for t, s in zip(self.target_shape, volume.shape)]
        resized = ndimage.zoom(volume, factors, order=1)
        mask = resized > 0
        if np.any(mask):
            mean = np.mean(resized[mask])
            std = np.std(resized[mask]) + 1e-8
            resized[mask] = (resized[mask] - mean) / std
        return resized

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        temp_extract_dir = os.path.join("/tmp", pid)
        os.makedirs(temp_extract_dir, exist_ok=True)

        try:
            with tarfile.open(self.tar_file_path, "r") as tf:
                tf.extractall(path=temp_extract_dir, members=self.patient_catalog[pid])

            patient_folder = os.path.join(temp_extract_dir, "brats2021", pid)
            if not os.path.exists(patient_folder):
                patient_folder = temp_extract_dir

            flair_files = glob.glob(os.path.join(patient_folder, "*_flair.nii.gz"))
            seg_files = glob.glob(os.path.join(patient_folder, "*_seg.nii.gz"))

            if not flair_files or not seg_files:
                return None

            flair_data = nib.load(flair_files[0]).get_fdata()
            seg_data = nib.load(seg_files[0]).get_fdata()

            mri_norm = self._resize_and_normalize(flair_data)
            mri_tensor = torch.tensor(mri_norm, dtype=torch.float32).unsqueeze(0)

            solid_voxels = int(np.sum((seg_data == 1) | (seg_data == 4)))
            has_edema = bool(np.sum(seg_data == 2) > 0)

            q_type = np.random.choice(["size", "sub_region_presence"])
            if q_type == "size":
                question = "What is the approximate solid tumor volume in this brain scan?"
                answer = f"The solid tumor volume is approximately {solid_voxels} cubic millimeters."
            else:
                question = "Is peritumoral edema present in this scan?"
                answer = "Yes, peritumoral edema is observed." if has_edema else "No peritumoral edema detected."

            return {
                "mri": mri_tensor,
                "question": question,
                "answer": answer,
                "question_type": q_type
            }
        except Exception:
            return None
        finally:
            if os.path.exists(temp_extract_dir):
                shutil.rmtree(temp_extract_dir, ignore_errors=True)

def safe_collate(batch):
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    return {
        "mri": torch.stack([b["mri"] for b in batch]),
        "question": [b["question"] for b in batch],
        "answer": [b["answer"] for b in batch],
        "question_type": [b["question_type"] for b in batch],
    }

# **Cell 3: Architecture Definition & Token Alignment**
*Connects frozen BrainIAC (3D ViT) and 4-bit LLaMA-3.1 8B via the trainable Adapter $g_i$, masking prompt loss with `-100` so gradients update strictly on generated answers.*

In [9]:
import torch
import torch.nn as nn
from monai.networks.nets import ViT
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class BrainTumorAdapter(nn.Module):
    """g_i: Trainable 3-layer FFN translator (Linear -> GELU -> Linear)"""
    def __init__(self, vision_dim=768, llm_dim=4096):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )
        
    def forward(self, z):
        return self.proj(z)

class BrainTumorVLM(nn.Module):
    def __init__(self, llm_id="meta-llama/Meta-Llama-3.1-8B"):
        super().__init__()
        
        # 1. Vision Encoder f_vision (3D ViT matching BrainIAC)
        print("🧠 Loading BrainIAC 3D ViT Vision Encoder...")
        self.vision_encoder = ViT(
            in_channels=1, 
            img_size=(96, 96, 96), 
            patch_size=(16, 16, 16), 
            hidden_size=768, 
            mlp_dim=3072, 
            num_layers=12, 
            num_heads=12,
            classification=False
        )
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
            
        # 2. Shared LLM (4-bit NF4 Quantization)
        print("🦙 Loading LLaMA-3.1 8B in 4-bit NF4...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(llm_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        for param in self.llm.parameters():
            param.requires_grad = False
            
        self.d_llm = self.llm.config.hidden_size # 4096
        
        # 3. Trainable Adapter (g_i)
        self.adapter = BrainTumorAdapter(vision_dim=768, llm_dim=self.d_llm)

    def forward(self, mri_scans, questions, answers, question_types):
        target_device = next(self.llm.parameters()).device
        target_dtype = torch.float16
        
        # Move vision components explicitly to target_device to resolve CPU/GPU mismatch
        self.vision_encoder.to(target_device)
        self.adapter.to(target_device)
        
        # Vision Pass: Z = f_vision(x)
        mri_scans = mri_scans.to(device=target_device, dtype=torch.float32)
        with torch.no_grad():
            vit_out = self.vision_encoder(mri_scans)
            if isinstance(vit_out, tuple): 
                vit_out = vit_out[0]
            z = vit_out # Shape: (Batch, 216, 768)
            
        # Adapter Translation: H_img = g_i(Z)
        h_img = self.adapter(z).to(dtype=target_dtype)
        batch_size, num_img_tokens, _ = h_img.shape
        
        # Tokenize Prompt & Full Text
        prompts = [f"Question: {q}\nAnswer: " for q in questions]
        full_texts = [f"Question: {q}\nAnswer: {a}" for q, a in zip(questions, answers)]
        
        prompt_enc = self.tokenizer(prompts, padding=True, return_tensors="pt").to(target_device)
        full_enc = self.tokenizer(full_texts, padding=True, return_tensors="pt").to(target_device)
        
        e_q = self.llm.get_input_embeddings()(full_enc.input_ids).to(dtype=target_dtype)
        
        # Combine Embeddings: H = [H_img ; E_q]
        h_combined = torch.cat([h_img, e_q], dim=1)
        
        img_mask = torch.ones((batch_size, num_img_tokens), device=target_device, dtype=torch.long)
        combined_mask = torch.cat([img_mask, full_enc.attention_mask], dim=1)
        
        # Construct Target Labels for Causal LM Loss
        labels = full_enc.input_ids.clone()
        labels[full_enc.attention_mask == 0] = -100
        for i in range(batch_size):
            prompt_len = prompt_enc.attention_mask[i].sum().item()
            labels[i, :prompt_len] = -100 # Mask prompt; train ONLY on answer
            
        img_labels = torch.full((batch_size, num_img_tokens), -100, device=target_device, dtype=torch.long)
        combined_labels = torch.cat([img_labels, labels], dim=1)
        
        outputs = self.llm(
            inputs_embeds=h_combined,
            attention_mask=combined_mask,
            labels=combined_labels
        )
        
        return outputs.loss

# **Cell 4: Memory-Managed Execution & Checkpointing**
*Executes training with periodic GPU cache clearing and saves the trained adapter checkpoint.*

In [10]:
from tqdm import tqdm

found_files = glob.glob("/kaggle/input/**/brats2021_processed.tar", recursive=True)
if not found_files:
    raise FileNotFoundError("brats2021_processed.tar missing. Click '+ Add Input' on Kaggle.")

dataset = BraTS_VLM_VirtualDataset(tar_file_path=found_files[0])
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=safe_collate, num_workers=0)

model = BrainTumorVLM()
optimizer = torch.optim.AdamW(model.adapter.parameters(), lr=1e-4, weight_decay=0.01)

num_epochs = 1
accumulation_steps = 2
print("\n🚀 Starting Production-Grade VLM Training Loop...")

model.adapter.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    valid_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        loss = model(batch["mri"], batch["question"], batch["answer"], batch["question_type"])
        loss = loss / accumulation_steps
        loss.backward()

        if (batch_idx + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            gc.collect()
            torch.cuda.empty_cache()

        running_loss += loss.item() * accumulation_steps
        valid_batches += 1
        pbar.set_postfix({"loss": f"{loss.item() * accumulation_steps:.4f}"})

    if valid_batches > 0:
        print(f"✅ Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss / valid_batches:.4f}")

# Save Checkpoint
checkpoint_path = "/kaggle/working/brain_tumor_adapter.pt"
torch.save(model.adapter.state_dict(), checkpoint_path)
print(f"🎉 Success! Adapter saved to: {checkpoint_path}")

🧠 Loading BrainIAC 3D ViT Vision Encoder...
🦙 Loading LLaMA-3.1 8B in 4-bit NF4...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


🚀 Starting Production-Grade VLM Training Loop...


Epoch 1/1:   0%|          | 0/624 [00:00<?, ?it/s]/tmp/ipykernel_58/3003584595.py:46: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(path=temp_extract_dir, members=self.patient_catalog[pid])
/tmp/ipykernel_58/3003584595.py:46: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(path=temp_extract_dir, members=self.patient_catalog[pid])
Epoch 1/1:   0%|          | 0/624 [00:07<?, ?it/s]


RuntimeError: expected mat1 and mat2 to have the same dtype, but got: float != c10::BFloat16